In [14]:
from handwriting_sample import HandwritingSample as hs

svc_sample = hs.from_svc(path="C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc")
print(svc_sample)



DEBUG:fsspec.local:open file: C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc
2025-10-26 21:19:01 - 272 - SVCFileReader - Old file-name format no additional meta data
2025-10-26 21:19:01 - 272 - SVCFileReader - Data has been loaded from an SVC file: C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc
<HandwritingSampleObject: 
DATA:
   x =          [239.91  239.91  239.91  ... 267.82  267.8   267.645], 
   y =          [141.41  141.41  141.41  ... 134.06  134.155 134.465], 
   time =       [0.0000e+00 7.0000e-03 1.5000e-02 ... 1.7539e+01 1.7547e+01 1.7554e+01], 
   pen_status = [ True  True  True ...  True  True  True], 
   azimuth =    [1390. 1390. 1400. ... 1420. 1420. 1420.], 
   tilt =       [560. 560. 560. ... 590. 590. 590.], 
   pressure =   [0.002933 0.025415 0.047898 ... 0.260997 0.246334 0.057674]> 


METADATA:
dict_items([('samples_count', 2336)])


In [2]:
from handwriting_features import HandwritingFeatures as hf

data_path = "C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc"

variables = ["y", "x", "time", "pen_status", "azimuth", "tilt", "pressure"]

fs = 133  # Sampling frequency in Hz

feature_data = hf.from_svc(data_path, variables)

# 1. Kinematic features
x_velocity = feature_data.velocity(axis="x", in_air=False, statistics=["mean", "std"])
y_velocity = feature_data.velocity(axis="y", in_air=False, statistics=["mean", "std"])

pressure = feature_data.pressure(statistics=["median", "std"])

print("X Velocity Features:", x_velocity)
print("Y Velocity Features:", y_velocity)
print("Pressure Features:", pressure)

DEBUG:fsspec.local:open file: C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc
2025-10-26 21:23:47 - 272 - SVCFileReader - Old file-name format no additional meta data
2025-10-26 21:23:47 - 272 - SVCFileReader - Data has been loaded from an SVC file: C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc
X Velocity Features: [23.58355521 15.80903653]
Y Velocity Features: [26.17234362 20.51260332]
Pressure Features: [0.173021   0.03585308]


In [ ]:
import os
from pathlib import Path
import pandas as pd
from handwriting_features import HandwritingFeatures as hf

directory_path = "C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002"

pathlist = Path(directory_path).rglob("*.svc")

variables = ["y", "x", "time", "pen_status", "azimuth", "tilt", "pressure"]
x_velocity, y_velocity, pressure= [],[],[]
pd_rows = []
fs = 133  # Sampling frequency in Hz
i = 0

def diagnosis_from_filename(filename):
    if filename.startswith("HC"):
        return 1
    elif filename.startswith("pre-LBD"):
        return 0
    else:
        return None  # or raise an exception if appropriate
for path in pathlist:
    feature_data = hf.from_svc(str(path), variables)
    diagnosis = diagnosis_from_filename(path.name)
    x_velocity_loop = feature_data.velocity(axis="x", in_air=False, statistics=["mean", "std"])
    y_velocity_loop = feature_data.velocity(axis="y", in_air=False, statistics=["mean", "std"])
    pressure_loop = feature_data.pressure(statistics=["mean", "std"])
    
    x_velocity.append(x_velocity_loop)
    y_velocity.append(y_velocity_loop)
    pressure.append(pressure_loop)

    pandas_row = {
        "file_name": path.name,
        "diagnosis": diagnosis,
        "x_velocity_mean" : x_velocity_loop[0],
        "x_velocity_std" : x_velocity_loop[1],
        "y_velocity_mean" : y_velocity_loop[0],
        "y_velocity_std" : y_velocity_loop[1],
        "pressure_mean" : pressure_loop[0],
        "pressure_std" : pressure_loop[1]
    }
    pd_rows.append(pandas_row)
    i = i + 1
    df = pd.DataFrame(pd_rows)

print("Processed files:", i)
print(f"File: {path.name}, Diagnosis: {diagnosis}")
print("X Velocity Features:", x_velocity_loop)
print("Y Velocity Features:", y_velocity_loop)
print("Pressure Features:", pressure_loop)







FileNotFoundError: [Errno 2] No such file or directory: 'HC-1#1_w.cz.fnusa.10_1.svc'

In [11]:
from pathlib import Path
import pandas as pd
from handwriting_features import HandwritingFeatures as HF

directory_path = Path("C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002")
variables = ["y", "x", "time", "pen_status", "azimuth", "tilt", "pressure"]
fs = 133  # Hz

def diagnosis_from_path(p: Path):
    # Look in filename first, then parent dirs
    names = [p.stem, p.name] + [par.name for par in p.parents]
    for name in names:
        if name.startswith("HC"):
            return 1          # Healthy control
        if name.startswith("pre-LBD"):
            return 0          # Patient (pre-LBD)
    return None

def extract_mean_std(stats):
    """
    Accepts either a dict like {'mean': m, 'std': s} or a tuple/list (m, s).
    Returns (mean, std) or (None, None) if missing.
    """
    if isinstance(stats, dict):
        return stats.get("mean"), stats.get("std")
    if isinstance(stats, (list, tuple)) and len(stats) >= 2:
        return stats[0], stats[1]
    return None, None

rows = []
processed = skipped = errors = 0

for path in directory_path.rglob("*.svc"):
    diag = diagnosis_from_path(path)
    if diag is None:
        skipped += 1
        continue

    try:
        # Build feature object for this file.
        # If your API differs (e.g., HF.from_svc(path, fs=..., variables=...)), swap this line.
        feature_data = HF(path, variables=variables, fs=fs)

        x_vel = feature_data.velocity(axis="x", in_air=False, statistics=["mean", "std"])
        y_vel = feature_data.velocity(axis="y", in_air=False, statistics=["mean", "std"])
        pres  = feature_data.pressure(statistics=["mean", "std"])

        xv_mean, xv_std = extract_mean_std(x_vel)
        yv_mean, yv_std = extract_mean_std(y_vel)
        p_mean,  p_std  = extract_mean_std(pres)

        rows.append({
            "file_name": path.name,
            "relpath": str(path.relative_to(directory_path)),
            "diagnosis": diag,
            "x_velocity_mean": xv_mean,
            "x_velocity_std":  xv_std,
            "y_velocity_mean": yv_mean,
            "y_velocity_std":  yv_std,
            "pressure_mean":   p_mean,
            "pressure_std":    p_std,
        })
        processed += 1

    except Exception as e:
        errors += 1
        # Optional quick debug:
        # print(f"ERR {path}: {e}")

# Build the wide table (not nested!)
df = pd.DataFrame(rows)

print(f"Processed files: {processed}")
print(f"Skipped (no diagnosis in path): {skipped}")
print(f"Errors: {errors}")
print(df.shape)
print(df.head())
# df.to_csv("lbd_features.csv", index=False)


Processed files: 0
Skipped (no diagnosis in path): 0
Errors: 1911
(0, 0)
Empty DataFrame
Columns: []
Index: []
